# 探究语义信息(熵)对 Top-K 和 阈值选择的影响

本 Notebook 基于 RL Adapter 的训练历史记录 (`.buffer` 文件)，分析模型在不同语义不确定性（Entropy）下如何选择 Speculative Decoding 的配置（Top-K 和 Threshold）。

虽然实验结果 JSON 文件包含宏观指标，但详细的每一步决策（包含熵值、动作选择）存储在 RL Agent 的 Replay Buffer 中。我们将加载该 Buffer 进行深入分析。

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# 配置绘图风格
sns.set_theme(style="whitegrid")
# plt.rcParams['font.sans-serif'] = ['SimHei']  #由于系统可能缺失中文字体，暂时注释掉。因为图表是英文的，使用默认字体即可。
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

In [ ]:
# 定义动作空间映射
TOPK_CANDIDATES = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
THRESHOLD_CANDIDATES = [0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99]

# 尝试加载多个可能的 Buffer 路径
potential_paths = [
    "checkpoints/llama/rl_adapter_main.pth.buffer",
    "checkpoints/llama/rl_adapter_little.pth.buffer",
    "checkpoints/rl_adapter_main.pth.buffer",
    "checkpoints/rl_adapter_little.pth.buffer"
]

memory = []
loaded_path = ""

for buffer_path in potential_paths:
    if os.path.exists(buffer_path):
        try:
            with open(buffer_path, 'rb') as f:
                loaded_mem = pickle.load(f)
                if len(loaded_mem) > 0:
                    memory = loaded_mem
                    loaded_path = buffer_path
                    print(f"Successfully loaded {len(memory)} transitions from {buffer_path}")
                    break
                else:
                    print(f"Found empty buffer at {buffer_path}")
        except Exception as e:
            print(f"Error loading {buffer_path}: {e}")

# 提取数据
data = []
if len(memory) > 0:
    for state_seq, action, reward, next_state_seq, done in memory:
        # state_seq shape: (seq_len, feature_dim)
        # 取最后一步的状态
        current_state = state_seq[-1]
        
        # feature_dim = [norm_bw, norm_lat, norm_entropy, last_acc, task_vec...]
        # Index 2 is norm_entropy = min(entropy / 10.0, 1.0)
        norm_entropy = current_state[2]
        entropy = norm_entropy * 10.0
        
        # 解析动作
        topk_idx = action // len(THRESHOLD_CANDIDATES)
        threshold_idx = action % len(THRESHOLD_CANDIDATES)
        
        selected_topk = TOPK_CANDIDATES[topk_idx]
        selected_threshold = THRESHOLD_CANDIDATES[threshold_idx]
        
        data.append({
            "Entropy": entropy,
            "TopK": selected_topk,
            "Threshold": selected_threshold,
            "Reward": reward
        })
else:
    print("WARNING: No valid training data found in buffers. Generating MOCK DATA for demonstration purposes.")
    # Generate mock data reflecting expected behaviors
    np.random.seed(42)
    for _ in range(500):
        # Mock entropy 0-10
        ent = np.random.gamma(2, 2) 
        ent = min(max(ent, 0), 10)
        
        # Simulating correlation: distinct behaviors for low vs high entropy
        if ent < 3:
            # Low entropy (Confident): 
            # 1. Very small K (confident simple tokens) OR 
            # 2. Very large K (confident bulk transmission)
            # High threshold (require high confidence)
            tk_idx = np.random.choice([0, 1, 2, 9, 10], p=[0.3, 0.2, 0.2, 0.15, 0.15])
            th_idx = np.random.choice([5, 6, 7], p=[0.2, 0.3, 0.5]) # High thresholds
        elif ent < 7:
            # Medium entropy
            tk_idx = np.random.choice([3, 4, 5, 6], p=[0.25, 0.25, 0.25, 0.25])
            th_idx = np.random.choice([3, 4, 5], p=[0.3, 0.4, 0.3])
        else:
            # High entropy (Uncertain): 
            # Try to grab more tokens (Medium-High K) but lower threshold to accept them
            tk_idx = np.random.choice([4, 5, 6, 7, 8], p=[0.1, 0.2, 0.4, 0.2, 0.1])
            th_idx = np.random.choice([0, 1, 2], p=[0.2, 0.5, 0.3]) # Low thresholds

        data.append({
            "Entropy": ent,
            "TopK": TOPK_CANDIDATES[tk_idx],
            "Threshold": THRESHOLD_CANDIDATES[th_idx],
            "Reward": np.random.random()
        })

df = pd.DataFrame(data)
print(f"DataFrame shape: {df.shape}")
df.head()

In [ ]:
# 1. 语义不确定性 (Entropy) 分布
plt.figure(figsize=(10, 6))
sns.histplot(df["Entropy"], bins=30, kde=True)
plt.title("Semantic Uncertainty (Entropy) Distribution")
plt.xlabel("Entropy")
plt.ylabel("Count")
plt.show()

# 2. 熵对 Top-K 选择的影响
# 将熵分为几个区间
df["Entropy_Bin"] = pd.cut(df["Entropy"], bins=[0, 1, 2, 4, 6, 8, 10], labels=["0-1", "1-2", "2-4", "4-6", "6-8", "8-10"])

plt.figure(figsize=(12, 6))
sns.boxplot(x="Entropy_Bin", y="TopK", data=df)
plt.yscale("log", base=2) # TopK 是指数增长的，用对数轴展示更清晰
plt.title("Impact of Semantic Uncertainty on Top-K Selection")
plt.xlabel("Entropy Range (Semantics)")
plt.ylabel("Selected Top-K (Log Scale)")
plt.show()

# 3. 熵对 Top-K 选择的平均趋势
plt.figure(figsize=(12, 6))
sns.lineplot(x="Entropy_Bin", y="TopK", data=df, marker="o")
plt.title("Average Top-K vs Semantic Uncertainty")
plt.xlabel("Entropy Range")
plt.ylabel("Average Top-K")
plt.show()

In [ ]:
# 4. 熵对 Threshold (置信度阈值) 选择的影响
plt.figure(figsize=(12, 6))
sns.boxplot(x="Entropy_Bin", y="Threshold", data=df)
plt.title("Impact of Semantic Uncertainty on Threshold Selection")
plt.xlabel("Entropy Range (Semantics)")
plt.ylabel("Selected Confidence Threshold")
plt.show()

# 5. 熵对 Threshold 选择的平均趋势
plt.figure(figsize=(12, 6))
sns.lineplot(x="Entropy_Bin", y="Threshold", data=df, marker="o", color="orange")
plt.title("Average Threshold vs Semantic Uncertainty")
plt.xlabel("Entropy Range")
plt.ylabel("Average Threshold")
plt.show()

## 结论分析：为什么熵越大，Top-K 反而越低？

通过上述分析，我们观察到一个有趣的现象：**当语义不确定性（熵）极高时，RL Agent 倾向于选择较小的 Top-K**。

这看起来反直觉（通常认为不确定时应该多选几个备选），但实际上是 RL 学出的一种**高效止损策略**：

1.  **避免“收益递减” (Diminishing Returns)**
    *   **低熵时**：概率分布尖锐，前几个词可能占据 99% 的可能性。增加一点 K 就能捕获所有可能，收益极高。
    *   **高熵时**：概率分布平坦，长尾效应显著。即使把 K 从 10 增加到 100（通信开销增加 10 倍），可能覆盖的概率总和只增加了 5%。
    *   **决策**：RL 发现为了这点微乎其微的命中率提升，付出巨大的传输延迟代价是**不划算 (Negative Reward)** 的。

2.  **极力压缩通信延迟 (Minimize Latency)**
    *   奖励函数大致与 TPS 成正比：$Reward \propto \frac{\text{Accepted Tokens}}{\text{Compute Time} + \text{Comm Time}}$。
    *   当熵很高时，预测极其困难，分子（Accepted Tokens）注定很低。
    *   **决策**：既然分子提不上去了，RL 只能通过极力压缩分母（Comm Time）来维持 TPS。**选择极小的 Top-K 是为了最小化网络传输时间**，即使被拒绝，系统也能迅速进入下一个 Token 的生成，而不是卡在传输大量无用的候选词上。

3.  **“放弃治療”策略**
    *   高熵意味着大模型自己也很犹豫。在这种情况下，小模型生成的候选词很有可能完全偏离大模型的想法。
    *   **决策**：既然怎么猜大概率都是错，不如直接放弃 Speculative Decoding 的“投机”部分，退化回标准解码（或接近标准解码的低负载模式），以节省带宽资源。

**总结**：模型学到了：**在极其不确定的情况下，与其花费昂贵的带宽去“广撒网”却依然捞不到鱼，不如直接放弃猜测，节省时间成本。**

### 补充分析：为什么熵越高，Threshold 也越低？

我们还观察到 **Entropy 越高 -> Threshold 越低** 的现象。这涉及到 `src/adapter.py` 中定义的停止机制（Stopping Mechanism）。

在本项目中，`Threshold` 用于控制 Draft Model 生成的停止条件：
- **停止条件**： `1 - 累计接受概率 > Threshold` (即 `累计拒绝概率 > Threshold`)。
- 当 **Threshold 越低**，意味着系统**更容易触发停止**。

在 **高熵（高不确定性）** 场景下：
1.  **Draft Model 信心不足**：每个 Token 的 Acceptance Probability (由 `acc_head` 预测) 都会比较低。
2.  **累积拒绝概率迅速上升**：因为单步成功率低，`1 - p1*p2*...` 会这非常快地接近 1。
3.  **RL 的应对**：为了避免生成大量注定被拒绝的 Token（浪费计算和传输资源），RL Agent 学会了 **降低 Threshold**。
    - **低 Threshold** = **更早停止**。
    - 在高熵环境下，模型意识到自己猜不对，因此与其硬猜长序列，不如赶紧停下来（Early Stopping），只生成并传输非常有把握的前几个 Token（或者干脆就是 Top-K 极低的情况）。

**总结：**
*   **低 Entropy**：模型自信 -> **High Threshold** (允许生成更长序列) + **High Top-K** (传输更多候选)。
*   **高 Entropy**：模型迷茫 -> **Low Threshold** (尽早停止生成) + **Low Top-K** (减少传输开销)。
这就是 RL 自动学到的 **“不确定就少做少错” (Conservative)** 策略。

In [ ]:
# 6. 综合图表：熵 vs Top-K & Threshold (学术论文风格)
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
from pathlib import Path

# 关键代码：这两行强制 Matplotlib 使用 TrueType (Type 42) 字体而不是 Type 3
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# 字体设置
rcParams["font.sans-serif"] = [
    "Noto Sans CJK SC",
    "Noto Sans CJK",
    "Source Han Sans SC",
    "WenQuanYi Micro Hei",
    "SimHei",
    "Arial Unicode MS",
    "DejaVu Sans",
    "sans-serif", 
]
rcParams["axes.unicode_minus"] = False

# 定义区间映射 (防止某些变量在前面未运行的单元格中定义)
bins = [0, 1, 2, 4, 6, 8, 10]
labels = ["0-1", "1-2", "2-4", "4-6", "6-8", "8-10"]
midpoints = [0.5, 1.5, 3.0, 5.0, 7.0, 9.0]
midpoint_map = dict(zip(labels, midpoints))

# 确保 DataFrame 有 Bin 列
if "Entropy_Bin" not in df.columns:
    df["Entropy_Bin"] = pd.cut(df["Entropy"], bins=bins, labels=labels)

# 准备数据
grouped_mean = df.groupby("Entropy_Bin", observed=True)[["TopK", "Threshold"]].mean()
valid_labels = grouped_mean.index.tolist()

# X 轴数值 (使用区间中点)
x_values = np.array([midpoint_map[label] for label in valid_labels])
y_topk = grouped_mean["TopK"].values
y_thresh = grouped_mean["Threshold"].values

# 排序
sorted_indices = np.argsort(x_values)
x_plot = x_values[sorted_indices]
y_topk_plot = y_topk[sorted_indices]
y_thresh_plot = y_thresh[sorted_indices]
labels_plot = [valid_labels[i] for i in sorted_indices]

# 颜色配置
blue_dark = "#1f77b4"   
pink_dark = "#e377c2"   
pink_light = "#f7b6d2" 

# 输出目录
output_dir = "plots"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# 绘图初始化
fig, ax1 = plt.subplots(figsize=(6, 4), dpi=300)

# 左轴：Top-K (Blue)
line_topk, = ax1.plot(x_plot, y_topk_plot, marker="o", linewidth=2, label="Avg Top-K", color=blue_dark)

ax1.set_xlabel("Entropy Range (Semantics)")
ax1.set_ylabel("Average Top-K", color=blue_dark) # 只有左轴标题设置颜色，或者跟随风格不设置颜色
# ax1.set_ylabel("Average Top-K") # 用户风格中左轴是 Accuracy 自然黑色，但 tick 是蓝色
ax1.set_xticks(x_plot)
ax1.set_xticklabels(labels_plot) # 显示区间标签
ax1.grid(True, linestyle=":", linewidth=0.8, alpha=0.7)
ax1.tick_params(axis="y", colors=blue_dark)
ax1.tick_params(axis="x", rotation=45) # 标签稍微旋转防止重叠

# 右轴：Threshold (Pink)
ax2 = ax1.twinx()
line_thresh, = ax2.plot(x_plot, y_thresh_plot, marker="^", linewidth=2, color=pink_dark, label="Avg Threshold")
ax2.set_ylabel("Threshold / Risk Tolerance", color=pink_dark) # 设置右轴标题颜色以区分
# ax2.set_ylabel("Threshold") 
ax2.tick_params(axis="y", colors=pink_dark)
ax2.set_ylim(0, 1.05)

# 图例 (如果需要)
# lines = [line_topk, line_thresh]
# headers = [l.get_label() for l in lines]
# ax1.legend(lines, headers, loc='upper center')
# 用户风格倾向于不使用图例，而是依靠轴颜色区分，这里保留简洁风格

fig.tight_layout()

# 保存
pdf_path = os.path.join(output_dir, "entropy_impact_style.pdf")
fig.savefig(pdf_path, bbox_inches="tight")

print(f"Saved: {pdf_path}")
plt.show()